# Under the Hood: What Python Is Actually Doing

You write `total = price * 1.08` and something answers. *What*, exactly? This lab lifts the floorboards: you will disassemble your own functions into the interpreter's instruction set, watch Python manage objects and memory, catch the call stack hitting its ceiling, and map the surprisingly tall tower of software your code stands on in this browser.

**How to use this notebook:** run cells top to bottom (`Shift+Enter`). A heads-up specific to this lab: several outputs (byte sizes, id numbers, bytecode listings) are *implementation details* — they legitimately vary between Python versions and machines, and this notebook runs Python on WebAssembly inside your browser. Where your numbers differ from a friend's, that IS one of the lessons.

Prerequisite: functions and lists; the machine lab (bits & bytes) helps.

## Bytecode: the interpreter's own language

Python does not execute your source text directly. First it **compiles** each function into **bytecode** — simple instructions for Python's internal *virtual machine* (a make-believe CPU implemented in software). The standard library's `dis` module (for *disassemble*) shows the bytecode of any function. Let's dissect one:

In [ ]:
import dis

def add_tax(price):
    total = price * 1.08
    return round(total, 2)

dis.dis(add_tax)

Reading the columns: leftmost is the source **line number**, then each instruction's byte offset, the **operation name**, and its argument (with a helpful comment in parentheses). The names are readable once you know the machine is stack-based — it shuffles values on and off an internal stack:

- `LOAD_FAST price` — push the local variable `price`;
- `LOAD_CONST 1.08` — push the constant;
- `BINARY_OP *` — pop two values, multiply, push the result;
- `STORE_FAST total` — pop into the local variable `total`;
- the `round(...)` call loads the function and its arguments, then a `CALL` instruction fires;
- `RETURN_VALUE` hands the result back.

(Exact opcode names shift a little between Python versions — another implementation detail on display.)

Every line of Python you have ever run was chewed into steps like these. Loops too — and counting instructions confirms why Python bothers with this form at all: it is much faster to *interpret* than raw text.

In [ ]:
import dis

def shout_three_times(word):
    for _ in range(3):
        print(word.upper() + "!")

instructions = list(dis.get_instructions(shout_three_times))
print(f"{len(instructions)} bytecode instructions, including:")
for ins in instructions[:8]:
    print(f"   {ins.opname:<20} {ins.argrepr}")

`dis.get_instructions` gives the same information as objects you can analyse with code — bytecode is data like everything else in Python. You'll use it in an exercise to settle a real question: does writing `x * x` compile differently from `x ** 2`?

## Names, objects, `id()` and `is`

Mental model time. In Python, **objects** live in memory, and **names** are labels tied to them. `id(obj)` returns a number that uniquely identifies an object during its lifetime (in CPython it is essentially the object's memory address), and `a is b` asks: *same object?* — not "equal value", but literally one object with two labels.

In [ ]:
a = [1, 2, 3]
b = a               # second label on the SAME object
c = list(a)         # a genuine copy: new object, equal contents

print("id(a):", id(a))
print("id(b):", id(b), "  <- same number as a")
print("id(c):", id(c), "  <- different object")

print("a is b:", a is b, "   a is c:", a is c)
print("a == c:", a == c, "   (equal contents, different boxes)")

`is` compares identity, `==` compares contents. The everyday rule: **use `==` for values; reserve `is` for `None`** (`if result is None:` — idiomatic because there is exactly one `None` object in the whole interpreter).

Now a genuinely strange experiment. Small integers feel like they should follow the same same-object/different-object logic... watch:

In [ ]:
x = 256
y = int("256")      # computed at runtime, so no compiler tricks
print("256:", x is y)

x = 257
y = int("257")
print("257:", x is y)

The value 256 gives `True` — one shared object — while 257 gives `False`. The explanation: CPython pre-creates the integers **-5 through 256** at startup (they are used constantly, so caching them saves memory and time), and every `256` in your program is a label on that one cached object. `257` is past the cache boundary, so each computation builds a fresh object.

This is a peek behind the curtain, not a feature to use: the cache range is an undocumented implementation detail. It's here because it perfectly demonstrates the names-vs-objects model — and because one day you'll see `is` mysteriously "work" for small numbers and "break" for big ones, and now you'll know why. **Compare values with `==`, always.**

## How lists grow: the over-allocation staircase

You append to a list ten thousand times, and each append is (amortised) $O(1)$. But *how*? If the list resized its memory block on every append, appends would be brutally slow. The trick: when a list runs out of room, it grabs **more space than it needs** — so the next several appends are free. `sys.getsizeof` lets us catch it in the act:

In [ ]:
import sys

items = []
previous_size = sys.getsizeof(items)
print(f"len  0: {previous_size} bytes")
for n in range(1, 33):
    items.append(n)
    size = sys.getsizeof(items)
    if size != previous_size:                 # only report the JUMPS
        print(f"len {n:>2}: {previous_size} -> {size} bytes  (room reserved!)")
        previous_size = size

The size does not tick up by one slot per append — it *jumps*, then stays flat while appends fill the reserved slots for free. Each flat stretch is spare capacity bought in advance. A picture makes the pattern unmistakable:

In [ ]:
import sys
import matplotlib.pyplot as plt

lengths, sizes = [], []
staircase = []
for n in range(129):
    lengths.append(len(staircase))
    sizes.append(sys.getsizeof(staircase))
    staircase.append(n)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.step(lengths, sizes, where="post")
ax.set_xlabel("list length (number of items)")
ax.set_ylabel("sys.getsizeof(list) in bytes")
ax.set_title("List over-allocation: memory grows in a staircase, not a ramp")
ax.grid(True, alpha=0.3)

A staircase, not a ramp. Each tread is a stretch of "free" appends; each riser is one real reallocation (allocate a bigger block, copy everything over). The steps get *wider* as the list grows — CPython over-allocates roughly an extra eighth — so the expensive copies become rarer exactly as they get more expensive. That balancing act is what "amortised $O(1)$ append" means in the flesh.

(Your exact byte numbers depend on the platform — under WebAssembly, pointers are 4 bytes instead of 8, so the whole staircase is skinnier than on a desktop Python. The *shape* is identical.)

## The garbage collector: who cleans up?

You create objects by the thousands and never free a single one. Someone must. CPython's primary mechanism is **reference counting**: every object carries a count of how many references point to it, and the moment the count hits zero the object is reclaimed, instantly.

In [ ]:
import sys

box = ["fragile"]
print("references:", sys.getrefcount(box))     # typically 2: `box` + the call's argument

alias = box
print("after alias:", sys.getrefcount(box))    # one more label -> one more

del alias
print("after del:  ", sys.getrefcount(box))    # back down

(`getrefcount` reports one *higher* than you'd guess, because passing `box` into the function creates a temporary extra reference. Implementation details, everywhere!)

Reference counting has one blind spot: **cycles**. If object A points to B and B points back to A, their counts never reach zero even when nothing else can reach them — undead memory. For exactly this case Python runs a backup **cycle collector**, exposed in the `gc` module. Let's manufacture a cycle, orphan it, and order a sweep:

In [ ]:
import gc

class Buddy:
    def __init__(self):
        self.partner = None

a = Buddy()
b = Buddy()
a.partner = b          # A -> B
b.partner = a          # B -> A : a reference cycle

del a, b               # no names left - but the cycle keeps both counts at 1

found = gc.collect()   # run the cycle detector NOW
print(f"cycle collector reclaimed {found} unreachable object(s)")
print("current gc counters (young, middle, old):", gc.get_count())

The collector found our orphaned pair (the count can exceed 2 — it sweeps up whatever other cyclic litter was around). Normally you never call `gc.collect()` yourself; it runs automatically when the counters you just printed cross thresholds. The takeaway is the division of labour: refcounting reclaims almost everything *immediately*, and the cycle collector patrols for the pathological leftovers.

## The call stack has a ceiling

Every function call pushes a **frame** (its local variables and bookkeeping) onto the **call stack**; every return pops one. Recursion stacks frames fast — and the stack is finite. Python enforces a limit to fail cleanly rather than crash the whole process:

In [ ]:
import sys
print("recursion limit:", sys.getrecursionlimit())

def dig(depth):
    try:
        return dig(depth + 1)           # keep descending...
    except RecursionError:
        return depth                    # ...until Python says stop

print("actual depth reached from here:", dig(0))

The measured depth comes in a bit under the limit — some frames were already on the stack before `dig` started (the notebook machinery that ran your cell!).

A runaway recursion — say, a base case you forgot — therefore doesn't hang forever; it dies with `RecursionError`. And because it's an ordinary exception, you can catch it like any other:

In [ ]:
def countdown_forever(n):
    return countdown_forever(n + 1)     # no base case: doomed

try:
    countdown_forever(0)
except RecursionError as err:
    print("Caught it:", err)
print("...and the notebook lives on.")

This is why deep recursion in Python is a design smell: an iterative loop uses one frame no matter how many repetitions. (There *is* a `sys.setrecursionlimit()`, but raising it just moves the cliff closer to a real crash — the honest fix is a loop.) One of the exercises has you perform exactly that rescue.

## The Pyodide tower: what your code is standing on

Time to zoom all the way out. This notebook feels like "just Python", but the machinery under this browser tab is a genuine tower:

1. **Your code** — `total = price * 1.08`.
2. **CPython compiles it to bytecode** — the `LOAD_FAST`/`BINARY_OP` instructions you disassembled with `dis`.
3. **The CPython interpreter executes that bytecode.** CPython is a large program written in C. Normally it's compiled to your machine's native instructions — but not here...
4. **...here CPython itself was compiled to WebAssembly** (this build is called **Pyodide**). WebAssembly ("wasm") is a portable, sandboxed instruction format that browsers agree to run — a make-believe CPU, standardised.
5. **The browser's WebAssembly engine** translates wasm into *actual* machine instructions, with the same security sandbox that contains every web page.
6. **The operating system and CPU** finally run those instructions on silicon.

So an interpreter (CPython) is running inside another virtual machine (wasm) inside a browser process managed by an OS on a processor. Two takeaways. First, this is why the byte sizes and speeds in this notebook differ from a desktop: wasm is a 32-bit world and adds interpretation overhead. Second — and more important — *nothing about this stacking is exotic*. Computing is layers all the way down; each layer only needs to honour its contract with the ones adjacent. You have now personally inspected the top three floors of the tower, which is more than most working programmers ever do.

## What you just learned

- Python compiles functions to bytecode; `dis` shows the stack-machine instructions.
- Names are labels on objects: `is`/`id()` reveal identity, `==` compares values — and CPython's small-int cache (-5..256) is a visible implementation detail, not a feature.
- Lists over-allocate in a staircase pattern, buying amortised $O(1)$ appends with spare capacity.
- Reference counting frees most objects instantly; the `gc` cycle collector handles reference cycles.
- The call stack is finite: recursion past `sys.getrecursionlimit()` raises a catchable `RecursionError`.
- Your code runs on a tower: Python → bytecode → CPython-in-WebAssembly → browser engine → OS → CPU.

## Try it yourself

### Exercise 1 — Bytecode detective

Do `x * x` and `x ** 2` compile to the same instructions? Disassemble both functions below with `dis.dis(...)`, then compare instruction *counts* with `len(list(dis.get_instructions(...)))`. Which operation names differ?

In [ ]:
import dis

def square_mul(x):
    return x * x

def square_pow(x):
    return x ** 2

# your code here: dis.dis both functions and compare
# len(list(dis.get_instructions(square_mul))) vs square_pow

### Exercise 2 — Aliasing, predicted

Apply the names-are-labels model: write your predictions for the two printed lists as comments **before** running the cell. One point per correct list; no partial credit from `id()` — reason it out first.

In [ ]:
team_a = ["Ana", "Ben"]
team_b = team_a            # label or copy?
team_c = list(team_a)      # label or copy?
team_b.append("Chi")
team_c.append("Dmitri")

# prediction: team_a -> ?
# prediction: team_c -> ?
print("team_a:", team_a)
print("team_c:", team_c)

### Exercise 3 — The dict staircase

Do dicts over-allocate like lists? Adapt the staircase experiment: add keys `0..128` to a dict one at a time, record `sys.getsizeof` at each length, and plot the step chart (axes labelled!). Does the dict staircase have wider or narrower treads than the list's?

In [ ]:
import sys
import matplotlib.pyplot as plt

lengths, sizes = [], []
growing = {}
# your code here: fill lengths/sizes while adding keys 0..128, then ax.step(...)
# Don't forget xlabel / ylabel / title.

print("points collected:", len(lengths))

### Exercise 4 — Rescue the recursive sum

`recursive_sum_to(100_000)` would blow the stack — try it if you like (you know how to catch the error now). Rewrite it as `loop_sum_to(n)` using an ordinary loop (or better: the closed form $n(n+1)/2$), then verify both agree where the recursive one still works.

In [ ]:
def recursive_sum_to(n):
    if n == 0:
        return 0
    return n + recursive_sum_to(n - 1)

def loop_sum_to(n):
    # your code here: one frame, any n
    pass

print("recursive, n=500:", recursive_sum_to(500))
# Uncomment to test the rescue:
# assert loop_sum_to(500) == recursive_sum_to(500)
# assert loop_sum_to(100_000) == 5_000_050_000
# print("loop_sum_to handles what recursion cannot!")

### Exercise 5 — Measure your tower

A tiny benchmark: time `sum(range(1_000_000))` here in the browser using `time.perf_counter()`. If you have a desktop Python installed, run the same line there and compare. How large is the WebAssembly toll on this particular operation — 1.5x, 3x, 10x? (There is no universally "right" answer: the toll depends on the workload. That's benchmarking's first lesson.)

In [ ]:
import time

start = time.perf_counter()
total = sum(range(1_000_000))
elapsed = time.perf_counter() - start
print(f"sum(range(1_000_000)) = {total}  in {elapsed * 1000:.2f} ms in THIS tower")

# your code here (optional): repeat the timing a few times - is the first run
# the slowest? Why might that be?